In [1]:
# 데이터 처리
import pandas as pd
import numpy as np
# URL 분석 / 문자열 패턴 정리
from urllib.parse import urlparse
import re
import math
# 머신러닝 데이터 분리
from sklearn.model_selection import train_test_split
# 모델: 랜덤 포레스트
from sklearn.ensemble import RandomForestClassifier
# 성능 평가
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 데이터셋 불러오기
import kagglehub
import os

In [2]:
path = kagglehub.dataset_download("sid321axn/malicious-urls-dataset")

print("Path to dataset files:", path)
print(os.listdir(path))

df = pd.read_csv(os.path.join(path, "malicious_phish.csv"))
df.head()

Using Colab cache for faster access to the 'malicious-urls-dataset' dataset.
Path to dataset files: /kaggle/input/malicious-urls-dataset
['malicious_phish.csv']


,url,type
0,br-icloud.com.br,phishing
1,mp3raid.com/music/krizz_kaliko.html,benign
2,bopsecrets.org/rexroth/cr/1.htm,benign
3,http://www.garage-pirenne.be/index.php?option=...,defacement
4,http://adventure-nicaragua.net/index.php?optio...,defacement


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 651191 entries, 0 to 651190
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   url     651191 non-null  object
 1   type    651191 non-null  object
dtypes: object(2)
memory usage: 9.9+ MB


In [4]:
df["type"].value_counts()

,count
type,
benign,428103
defacement,96457
phishing,94111
malware,32520


In [5]:
# 라벨 변환
df["label"] = df["type"].apply(lambda x: 0 if x == "benign" else 1)
df[["url", "type", "label"]].head() # 5개 출력

,url,type,label
0,br-icloud.com.br,phishing,1
1,mp3raid.com/music/krizz_kaliko.html,benign,0
2,bopsecrets.org/rexroth/cr/1.htm,benign,0
3,http://www.garage-pirenne.be/index.php?option=...,defacement,1
4,http://adventure-nicaragua.net/index.php?optio...,defacement,1


In [6]:
# URL 문자열의 특징들을 숫자로 나타내는 함수 구현
def normalize_for_parse(url):
    url = str(url).strip()

    if not url.startswith(("http://", "https://")):
        return "http://" + url

    return url


def safe_parse(url):
    parsed_url = normalize_for_parse(url)

    try:
        return urlparse(parsed_url)
    except ValueError:
        return None


def has_ip_address(url):
    pattern = r'(\d{1,3}\.){3}\d{1,3}'
    return 1 if re.search(pattern, str(url)) else 0


def split_tokens(text):
    """
    논문에서 hostname과 path token을 구분해 사용한 것처럼,
    URL 문자열을 여러 구분자로 나눔.
    """
    text = str(text).lower()
    tokens = re.split(r'[\/\?\.\=\-\_\&\%\:\s]+', text)
    return [token for token in tokens if token]


def get_domain_info(hostname):
    hostname = str(hostname).lower()

    # user:pass@host 형태가 있을 경우 @ 뒤쪽만 사용
    if "@" in hostname:
        hostname = hostname.split("@")[-1]

    # port 제거
    hostname = hostname.split(":")[0]

    parts = hostname.split(".") if hostname else []

    tld = parts[-1] if len(parts) >= 2 else ""
    primary_domain = ".".join(parts[-2:]) if len(parts) >= 2 else hostname
    subdomain_count = max(len(parts) - 2, 0)

    return tld, primary_domain, subdomain_count


def calculate_entropy(text):
    """
    랜덤 문자열처럼 보이는 URL을 잡기 위한 보조 feature.
    예: xj3k9qz-login-verify.com 같은 URL
    """
    text = str(text)

    if len(text) == 0:
        return 0

    probabilities = [text.count(c) / len(text) for c in set(text)]
    entropy = -sum(p * math.log2(p) for p in probabilities)

    return entropy


def extract_features(url):
    url = str(url).strip()
    parsed = safe_parse(url)

    if parsed is None:
        hostname = ""
        path = url
        query = ""
    else:
        hostname = parsed.netloc
        path = parsed.path
        query = parsed.query

    tld, primary_domain, subdomain_count = get_domain_info(hostname)

    hostname_tokens = split_tokens(hostname)
    path_tokens = split_tokens(path)
    query_tokens = split_tokens(query)
    all_tokens = hostname_tokens + path_tokens + query_tokens

    suspicious_words = [
        "login", "verify", "account", "secure", "update",
        "bank", "password", "signin", "confirm", "webscr",
        "paypal", "ebay", "free", "bonus", "token"
    ]

    special_char_count = sum(not c.isalnum() for c in url)
    digit_count = sum(c.isdigit() for c in url)

    url_length = len(url)
    hostname_length = len(hostname)
    path_length = len(path)
    query_length = len(query)

    token_lengths = [len(token) for token in all_tokens]

    features = {
        # 논문 기반 기본 lexical features
        "url_length": url_length,
        "hostname_length": hostname_length,
        "path_length": path_length,
        "query_length": query_length,

        # domain 구조
        "tld_length": len(tld),
        "primary_domain_length": len(primary_domain),
        "subdomain_count": subdomain_count,

        # 문자/구분자 개수
        "count_dot": url.count("."),
        "count_slash": url.count("/"),
        "count_hyphen": url.count("-"),
        "count_underscore": url.count("_"),
        "count_at": url.count("@"),
        "count_question": url.count("?"),
        "count_equal": url.count("="),
        "count_ampersand": url.count("&"),
        "count_percent": url.count("%"),

        # 숫자/특수문자 비율
        "count_digits": digit_count,
        "digit_ratio": digit_count / url_length if url_length > 0 else 0,
        "special_char_count": special_char_count,
        "special_char_ratio": special_char_count / url_length if url_length > 0 else 0,

        # token 기반 features
        "hostname_token_count": len(hostname_tokens),
        "path_token_count": len(path_tokens),
        "query_token_count": len(query_tokens),
        "total_token_count": len(all_tokens),
        "avg_token_length": sum(token_lengths) / len(token_lengths) if token_lengths else 0,
        "max_token_length": max(token_lengths) if token_lengths else 0,

        # path/query 구조
        "path_depth": path.count("/"),
        "query_param_count": query.count("="),

        # 악성 URL에서 자주 보이는 패턴
        "has_ip": has_ip_address(url),
        "contains_http_in_path": 1 if "http" in path.lower() else 0,
        "suspicious_word_count": sum(1 for word in suspicious_words if word in url.lower()),

        # 랜덤성 보조 지표
        "url_entropy": calculate_entropy(url),
        "hostname_entropy": calculate_entropy(hostname),
    }

    return features

In [7]:
# URL 전체를 숫자 특징으로 바꾸기
features = df["url"].apply(extract_features)

# 숫자들의 특징을 나타내는 표 생성
X = pd.DataFrame(features.tolist())
X.head()

,url_length,hostname_length,path_length,query_length,tld_length,primary_domain_length,subdomain_count,count_dot,count_slash,count_hyphen,...,total_token_count,avg_token_length,max_token_length,path_depth,query_param_count,has_ip,contains_http_in_path,suspicious_word_count,url_entropy,hostname_entropy
0,16,16,0,0,2,6,1,2,0,1,...,4,3.250000,6,0,0,0,0,0,3.375000,3.375000
1,35,11,24,0,3,11,0,2,2,0,...,6,5.000000,7,2,0,0,0,0,4.079143,3.277613
2,31,14,17,0,3,14,0,2,3,0,...,6,4.333333,10,3,0,0,0,0,3.708093,3.235926
3,88,21,10,49,2,17,1,3,3,1,...,16,4.125000,7,1,4,0,0,0,4.660343,3.308751
4,235,23,10,194,3,23,0,2,3,1,...,12,18.083333,156,1,3,0,0,0,5.491293,3.501398


In [8]:
# 정답 라벨 생성
y = df["label"]
y.head()

,label
0,1
1,0
2,0
3,1
4,1


In [9]:
# X 확인
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 651191 entries, 0 to 651190
Data columns (total 33 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   url_length             651191 non-null  int64  
 1   hostname_length        651191 non-null  int64  
 2   path_length            651191 non-null  int64  
 3   query_length           651191 non-null  int64  
 4   tld_length             651191 non-null  int64  
 5   primary_domain_length  651191 non-null  int64  
 6   subdomain_count        651191 non-null  int64  
 7   count_dot              651191 non-null  int64  
 8   count_slash            651191 non-null  int64  
 9   count_hyphen           651191 non-null  int64  
 10  count_underscore       651191 non-null  int64  
 11  count_at               651191 non-null  int64  
 12  count_question         651191 non-null  int64  
 13  count_equal            651191 non-null  int64  
 14  count_ampersand        651191 non-nu

In [10]:
X.shape

(651191, 33)

In [12]:
# 특징 추출
df_model = df.drop_duplicates(subset=["url"]).reset_index(drop=True)

features = df_model["url"].apply(extract_features)

X = pd.DataFrame(features.tolist())
y = df_model["label"]

X.head()

,url_length,hostname_length,path_length,query_length,tld_length,primary_domain_length,subdomain_count,count_dot,count_slash,count_hyphen,...,total_token_count,avg_token_length,max_token_length,path_depth,query_param_count,has_ip,contains_http_in_path,suspicious_word_count,url_entropy,hostname_entropy
0,16,16,0,0,2,6,1,2,0,1,...,4,3.250000,6,0,0,0,0,0,3.375000,3.375000
1,35,11,24,0,3,11,0,2,2,0,...,6,5.000000,7,2,0,0,0,0,4.079143,3.277613
2,31,14,17,0,3,14,0,2,3,0,...,6,4.333333,10,3,0,0,0,0,3.708093,3.235926
3,88,21,10,49,2,17,1,3,3,1,...,16,4.125000,7,1,4,0,0,0,4.660343,3.308751
4,235,23,10,194,3,23,0,2,3,1,...,12,18.083333,156,1,3,0,0,0,5.491293,3.501398


In [13]:
print(X.shape)
print(y.shape)

(641119, 33)
(641119,)


In [14]:
# RandomForest 모델 학습

# 학습용, 테스트용 분류: 학습용 80%, 테스트용 20%
X_train, X_test, y_train, y_test = train_test_split (
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 잘 분류되었는지 확인
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(512895, 33)
(128224, 33)
(512895,)
(128224,)


In [15]:
# Random Forest 모델 만들고 학습
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

In [16]:
# 테스트 데이터 예측
y_pred = model.predict(X_test)

In [17]:
# 성능 평가 출력
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9733279261292738

Confusion Matrix:
[[84322  1294]
 [ 2126 40482]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     85616
           1       0.97      0.95      0.96     42608

    accuracy                           0.97    128224
   macro avg       0.97      0.97      0.97    128224
weighted avg       0.97      0.97      0.97    128224



In [18]:
# 악성 URL 판별 함수 구현
def predict_url(url):
    feature = extract_features(url)
    feature_df = pd.DataFrame([feature])

    pred = model.predict(feature_df)[0]
    prob = model.predict_proba(feature_df)[0]

    if pred == 0:
        print("예측 결과: 정상 URL")
    else:
        print("예측 결과: 악성 URL")

    print(f"정상 확률: {prob[0]:.4f}")
    print(f"악성 확률: {prob[1]:.4f}")

In [19]:
predict_url("https://www.naver.com")

예측 결과: 악성 URL
정상 확률: 0.0300
악성 확률: 0.9700


In [20]:
predict_url("http://192.168.0.1/login/verify-account")

예측 결과: 악성 URL
정상 확률: 0.0200
악성 확률: 0.9800


In [21]:
predict_url("http://secure-login-account-verification.com/update")

예측 결과: 악성 URL
정상 확률: 0.1300
악성 확률: 0.8700
